# 02 · Train the custom classifier

Builds a Document Intelligence **custom classification model** from the blob
container populated in notebook `01`. Two classes (`invoice`, `claim`) are supplied
as separate blob prefixes through a short-lived user-delegation container SAS.

This follows [Custom classification model - Document Intelligence](https://learn.microsoft.com/en-us/azure/ai-services/document-intelligence/train/custom-classifier?view=doc-intel-4.0.0).
For SDK/API training, each source PDF also needs its raw Layout API response in a
matching `.ocr.json` sidecar; this notebook creates any missing sidecars.

Training is a long-running operation - expect a few minutes.

## 1. Setup

In [ ]:
import importlib
import sys
from pathlib import Path

_here = Path.cwd()
for _c in (_here, _here / 'notebooks', *_here.parents):
    if (_c / '_lib.py').exists():
        sys.path.insert(0, str(_c))
        break
import _lib

importlib.reload(_lib)
_lib.load_env()
admin = _lib.get_admin_client()

 admin client | endpoint=https://ais-insurance-workshop-dev-01.cognitiveservices.azure.com/
 auth = DefaultAzureCredential


## 2. Prepare and describe the classes

Classifier training uses each class's blob prefix through `AzureBlobContentSource`.
The container URL includes a short-lived `rwdl` user-delegation SAS, as shown in the
Microsoft Learn request. Missing raw Layout responses are generated first.

In [ ]:
from azure.ai.documentintelligence.models import (
    AzureBlobContentSource,
    BuildDocumentClassifierRequest,
    ClassifierDocumentTypeDetails,
)

layout_counts = _lib.ensure_classifier_layout_results()
training_container_sas = _lib.container_sas_url()
classifier_id = _lib.CLASSIFIER_ID

doc_types = {
    label: ClassifierDocumentTypeDetails(
        azure_blob_source=AzureBlobContentSource(
            container_url=training_container_sas,
            prefix=f'{label}/',
        )
    )
    for label in _lib.CLASS_DIRS
}

print('classifier id:', classifier_id)
print('container    :', _lib.container_url())
for label, details in doc_types.items():
    print(f'  {label:8s} <- {details.azure_blob_source.prefix}')

classifier id: insurance-invoice-claim-v1
  invoice  <- https://staidemosclsfrdev01.blob.core.windows.net/classifier-training/invoice/
  claim    <- https://staidemosclsfrdev01.blob.core.windows.net/classifier-training/claim/


## 3. Build

Any existing classifier with the same id is deleted first so the rebuild starts
clean (there is no clean holdout with only 5 invoices, so we always retrain).

In [3]:
_lib.delete_classifier_if_exists(admin, classifier_id)

print('building classifier — this can take a few minutes ...')
poller = admin.begin_build_classifier(
    BuildDocumentClassifierRequest(
        classifier_id=classifier_id,
        description='Insurance document classifier: invoice vs claim',
        doc_types=doc_types,
    )
)
classifier = poller.result()
print('done.')

building classifier — this can take a few minutes ...


HttpResponseError: (InvalidRequest) Invalid request.
Code: InvalidRequest
Message: Invalid request.
Exception Details:	(TrainingContentMissing) Training data is missing: Could not find any training data at the given path.
	Code: TrainingContentMissing
	Message: Training data is missing: Could not find any training data at the given path.

## 4. Inspect the trained model

In [ ]:
print('classifier id :', classifier.classifier_id)
print('api version   :', classifier.api_version)
print('description   :', classifier.description)
print('created       :', classifier.created_date_time)
print('expires       :', classifier.expiration_date_time)
print('doc types     :')
for dt in classifier.doc_types:
    print('   -', dt)

Next: classify documents — [`03_classify_from_storage.ipynb`](03_classify_from_storage.ipynb)
(from Blob Storage) or [`04_classify_local_file.ipynb`](04_classify_local_file.ipynb)
(individual local files).